> **About this notebook**
>
> This notebook is part of the example gallery of [CLABTOOLKIT](https://github.com/connectomicslab/clabtoolkit), an open-source Python toolkit for routine processing, manipulation and visualization of neuroimaging data. It shows how to use the `networktools` module.
>
> 📦 **Install:** `pip install clabtoolkit` ([PyPI](https://pypi.org/project/clabtoolkit/)) · 📖 **Docs:** [clabtoolkit.readthedocs.io](https://clabtoolkit.readthedocs.io) · 📄 **Paper:** [arXiv:2607.02638](https://arxiv.org/abs/2607.02638)
>
> **Author:** Yasser Alemán-Gómez (Connectomics Lab, Radiology Department, CHUV – Lausanne University Hospital)

## Testing Methods in the `NETWORKTOOLS` Module

This Jupyter Notebook serves as a platform to rigorously test the various methods included in the Python module `NETWORKTOOLS`.

This module is particularly valuable for researchers working with graph representations of neuroimaging
data: building CSR (Compressed Sparse Row) graphs from dense adjacency matrices, triangulated surface
meshes or edge lists, and detecting the connected components of those graphs.

The tests are organized into the following sections for clarity and ease of navigation:

### Table of Contents

---

* 🔷 1. [`CSR Graph Creation`](#network-csr_creation). Methods dedicated to create CSR graphs from different sources
    * 🔸 1.1. [`adjacency_matrix_to_csr`](#network-adjacency_matrix_to_csr). Convert a dense adjacency matrix to CSR (Compressed Sparse Row) format
    * 🔸 1.2. [`triangulated_mesh_to_csr`](#network-triangulated_mesh_to_csr). Convert triangulated mesh faces to a CSR graph representation
    * 🔸 1.3. [`edges_to_csr`](#network-edges_to_csr). Convert an edge list with values to a CSR graph representation

* 🔷 2. [`Connected Components`](#network-components). Methods dedicated to detect connected components on graphs
    * 🔸 2.1. [`edges_to_components`](#network-edges_to_components). Compute connected components from an edge array of arbitrary vertex indices
    * 🔸 2.2. [`connected_components`](#network-connected_components). Find connected components in a CSR graph representation

---

---
## <a id="network-csr_creation"></a>1. CSR Graph Creation
Methods dedicated to create CSR graphs from different sources.

### 1.1 `adjacency_matrix_to_csr`
<a id="network-adjacency_matrix_to_csr"></a>

Convert a dense adjacency matrix to CSR (Compressed Sparse Row) format.

In [ ]:
########################### Testing adjacency_matrix_to_csr ###########################
import clabtoolkit.networktools as cltnet
import numpy as np

print("Test 1: Converting a binary adjacency matrix of a 4-vertex graph to CSR format...")
adj = np.array([[0, 1, 1, 0],
                [1, 0, 1, 1],
                [1, 1, 0, 1],
                [0, 1, 1, 0]])
csr_graph = cltnet.adjacency_matrix_to_csr(adj)
print(f"  Shape                  : {csr_graph.shape}")
print(f"  Number of stored edges : {csr_graph.nnz}")
print(f"  data    : {csr_graph.data}")
print(f"  indices : {csr_graph.indices}")
print(f"  indptr  : {csr_graph.indptr}")
print("-------------------------------")
print(" ")

print("Test 2: Converting a sparse, weighted connectivity matrix (100 regions) and comparing the memory usage...")
rng = np.random.default_rng(seed=42)
conn = rng.random((100, 100))
conn = (conn + conn.T) / 2                   # Symmetric weights
conn[conn < 0.9] = 0                         # Keep only the strongest connections
np.fill_diagonal(conn, 0)
csr_conn = cltnet.adjacency_matrix_to_csr(conn)
csr_bytes = csr_conn.data.nbytes + csr_conn.indices.nbytes + csr_conn.indptr.nbytes
print(f"  Non-zero connections : {csr_conn.nnz} ({100 * csr_conn.nnz / conn.size:.1f}% density)")
print(f"  Dense matrix memory  : {conn.nbytes} bytes")
print(f"  CSR matrix memory    : {csr_bytes} bytes")
print(f"  Same values          : {np.allclose(csr_conn.toarray(), conn)}")
print("-------------------------------")
print(" ")

print("Test 3: Checking the validation of the input...")
for name, bad_input in [("Non-square matrix", np.zeros((3, 4))),
                        ("3D array", np.zeros((3, 3, 3))),
                        ("Python list", [[0, 1], [1, 0]])]:
    try:
        cltnet.adjacency_matrix_to_csr(bad_input)
    except (ValueError, TypeError) as e:
        print(f"  {name:18s} -> {type(e).__name__}: {e}")
print("-------------------------------")

### 1.2 `triangulated_mesh_to_csr`
<a id="network-triangulated_mesh_to_csr"></a>

Convert triangulated mesh faces to a CSR graph representation.

In [ ]:
########################### Testing triangulated_mesh_to_csr ###########################
import clabtoolkit.networktools as cltnet
import clabtoolkit.surfacetools as cltsurf
import numpy as np

print("Test 1: Building the graph of a tetrahedron (4 faces, 4 vertices)...")
faces = np.array([[0, 1, 2],
                  [0, 1, 3],
                  [0, 2, 3],
                  [1, 2, 3]])
csr_graph = cltnet.triangulated_mesh_to_csr(faces)
print(csr_graph.toarray())
print("-------------------------------")
print(" ")

print("Test 2: Building the graph of a single triangle with an explicit number of vertices...")
print("NOTE: Vertices 3 and 4 are not used by any face, so they remain isolated.")
csr_triangle = cltnet.triangulated_mesh_to_csr(np.array([[0, 1, 2]]), n_vertices=5)
print(f"  Shape: {csr_triangle.shape}")
print(csr_triangle.toarray())
print("-------------------------------")
print(" ")

print("Test 3: Building the 1-ring neighborhood graph of a simulated surface mesh...")
surf = cltsurf.Surface.simulate_surface(seed=42, shape="sphere")
vertices = surf.get_vertices()
faces = surf.get_faces()
mesh_graph = cltnet.triangulated_mesh_to_csr(faces, n_vertices=len(vertices))
degrees = np.diff(mesh_graph.indptr)        # Number of neighbors of each vertex
print(f"  Vertices: {len(vertices)}, Faces: {len(faces)}, Edges: {mesh_graph.nnz // 2}")
print(f"  Neighbors per vertex -> min: {degrees.min()}, mean: {degrees.mean():.2f}, max: {degrees.max()}")
print(f"  Neighbors of vertex 10: {mesh_graph.indices[mesh_graph.indptr[10]:mesh_graph.indptr[11]]}")
print("-------------------------------")

### 1.3 `edges_to_csr`
<a id="network-edges_to_csr"></a>

Convert an edge list with values to a CSR graph representation.

In [ ]:
########################### Testing edges_to_csr ###########################
import clabtoolkit.networktools as cltnet
import numpy as np

print("Test 1: Building an undirected weighted graph from an edge list...")
edges = np.array([[0, 1],
                  [1, 2],
                  [0, 2]])
values = np.array([2.5, 1.0, 3.2])
csr_graph = cltnet.edges_to_csr(edges, values)
print(csr_graph.toarray())
print("-------------------------------")
print(" ")

print("Test 2: Building a directed graph (symmetric=False) and a binary graph (no edge values)...")
edges_directed = np.array([[0, 1], [1, 2]])
csr_directed = cltnet.edges_to_csr(edges_directed, np.array([0.8, 1.5]), symmetric=False)
print("  Directed graph:")
print(csr_directed.toarray())
csr_binary = cltnet.edges_to_csr(edges_directed, n_vertices=4)
print("  Binary graph with 4 vertices (all the edge values are set to 1):")
print(csr_binary.toarray())
print("-------------------------------")
print(" ")

print("Test 3: Handling duplicated edges (their values are summed)...")
edges_dup = np.array([[0, 1], [0, 1], [1, 0]])
values_dup = np.array([1.0, 2.0, 0.5])
csr_dup = cltnet.edges_to_csr(edges_dup, values_dup)
print(csr_dup.toarray())
print("-------------------------------")

---
## <a id="network-components"></a>2. Connected Components
Methods dedicated to detect connected components on graphs.

### 2.1 `edges_to_components`
<a id="network-edges_to_components"></a>

Compute connected components from an edge array of arbitrary vertex indices.

In [ ]:
########################### Testing edges_to_components ###########################
import clabtoolkit.networktools as cltnet
import clabtoolkit.surfacetools as cltsurf
import numpy as np

print("Test 1: Finding the connected components of an edge list with non-contiguous vertex indices...")
edges = np.array([[10, 11], [11, 12],
                  [50, 51],
                  [100, 101], [101, 102], [102, 103]])
n_components, labels, sizes = cltnet.edges_to_components(edges)
print("  Labels (column 0: original vertex index, column 1: component label):")
print(labels)
print("-------------------------------")
print(" ")

print("Test 2: Getting the vertices of the largest component without printing the summary...")
n_components, labels, sizes = cltnet.edges_to_components(edges, verbose=False)
largest = labels[labels[:, 1] == 0, 0]
print(f"  Number of components : {n_components}")
print(f"  Component sizes      : {sizes}")
print(f"  Largest component    : {largest}")
print("-------------------------------")
print(" ")

print("Test 3: Finding the disconnected patches of a surface mesh...")
print("NOTE: Only the faces close to the two poles of a simulated sphere are kept.")
surf = cltsurf.Surface.simulate_surface(seed=42, shape="sphere")
vertices = surf.get_vertices()
faces = surf.get_faces()
z = vertices[:, 2] / np.abs(vertices[:, 2]).max()
keep = np.all(np.abs(z[faces]) > 0.5, axis=1)
patch_faces = faces[keep]
patch_edges = np.vstack([patch_faces[:, [0, 1]], patch_faces[:, [1, 2]], patch_faces[:, [0, 2]]])
n_components, labels, sizes = cltnet.edges_to_components(patch_edges)
print("-------------------------------")

### 2.2 `connected_components`
<a id="network-connected_components"></a>

Find connected components in a CSR graph representation.

In [ ]:
########################### Testing connected_components ###########################
import clabtoolkit.networktools as cltnet
import clabtoolkit.surfacetools as cltsurf
import numpy as np
import warnings

print("Test 1: Finding the components of a graph with 3 components: [0, 1], [2, 3, 4] and [5]...")
edges = np.array([[0, 1], [2, 3], [2, 4], [3, 4]])
graph = cltnet.edges_to_csr(edges, n_vertices=6)
n_components, labels, sizes = cltnet.connected_components(graph)
print(f"  Component label of each vertex: {labels[:, 1]}")
print("-------------------------------")
print(" ")

print("Test 2: Finding the components of a mesh built from two separate surfaces...")
surf = cltsurf.Surface.simulate_surface(seed=42, shape="sphere")
n_vert = len(surf.get_vertices())
faces = surf.get_faces()
two_meshes = np.vstack([faces, faces + n_vert])            # Second copy with shifted vertex indices
mesh_graph = cltnet.triangulated_mesh_to_csr(two_meshes, n_vertices=2 * n_vert + 3)  # + 3 isolated vertices
n_components, labels, sizes = cltnet.connected_components(mesh_graph)
print("NOTE: The 3 vertices not used by any face are reported as single-vertex components.")
print("-------------------------------")
print(" ")

print("Test 3: Finding the weakly connected components of a directed graph...")
print("NOTE: A warning is raised and the edges are treated as undirected.")
directed = cltnet.edges_to_csr(np.array([[0, 1], [1, 2], [3, 4]]), symmetric=False)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    n_components, labels, sizes = cltnet.connected_components(directed, verbose=False)
for w in caught:
    print(f"  Warning: {w.message}")
print(f"  Number of components : {n_components}")
print(f"  Component sizes      : {sizes}")
print("-------------------------------")